# 04 - Geometry, Simulation Truth, And Lateral Distance

This notebook is about connecting three ideas:

1. The GCD file tells us where DOMs are.
2. The simulation file can contain a true particle track.
3. We can calculate how far each pulsed DOM is from that track.

The calculation is written out in ordinary Python so you can see each step.


In [ ]:
from pathlib import Path
from math import sin, cos

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from icecube import dataio, dataclasses

SIM_GCD = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133575/Level2_IC86.2019_data_Run00133575_0101_78_503_GCD.i3.zst')
SIM_FILE = Path('/data/sim/IceCube/2020/filtered/level2/CORSIKA-in-ice/20904/0000000-0000999/Level2_IC86.2020_corsika.020904.000000.i3.zst')

print('Simulation GCD exists:', SIM_GCD.exists())
print('Simulation file exists:', SIM_FILE.exists())


## Read the geometry from the GCD file

The geometry frame contains `I3Geometry`. Inside it, `geometry.omgeo` maps each OMKey to a DOM position. For this tutorial we keep standard in-ice DOMs and ignore IceTop.


In [ ]:
def frame_stop(frame):
    names = {'G': 'Geometry', 'C': 'Calibration', 'D': 'DetectorStatus', 'Q': 'DAQ', 'P': 'Physics', 'I': 'TrayInfo'}
    return names.get(str(frame.Stop), str(frame.Stop))

def read_geometry(gcd_path):
    gcd_file = dataio.I3File(str(gcd_path))
    while gcd_file.more():
        frame = gcd_file.pop_frame()
        if frame_stop(frame) == 'Geometry' and 'I3Geometry' in frame:
            geometry = frame['I3Geometry']
            gcd_file.close()
            return geometry
    gcd_file.close()
    raise RuntimeError('No I3Geometry found in the GCD file.')

def is_inice_dom(omkey):
    return 1 <= omkey.string <= 86 and 1 <= omkey.om <= 60

def dom_position_xyz(geometry, omkey):
    position = geometry.omgeo[omkey].position
    return np.array([position.x, position.y, position.z], dtype=float)

geometry = read_geometry(SIM_GCD)
inice_positions = {omkey: dom_position_xyz(geometry, omkey) for omkey in geometry.omgeo.keys() if is_inice_dom(omkey)}
print('Number of in-ice DOM positions loaded:', len(inice_positions))


## Plot the detector from above

This scatter plot is not event data. It is the detector geometry: the x-y positions of in-ice DOMs.


In [ ]:
geometry_rows = []
for omkey, position in inice_positions.items():
    geometry_rows.append({'string': omkey.string, 'om': omkey.om, 'x': position[0], 'y': position[1], 'z': position[2]})

geometry_table = pd.DataFrame(geometry_rows)
ax = geometry_table.plot.scatter(x='x', y='y', s=4, alpha=0.4, figsize=(6, 6))
ax.set_aspect('equal')
ax.set_title('InIce DOM positions, viewed from above')
print('Each dot is one in-ice DOM position from the GCD file.')


## Pulse and truth helpers

The next functions do three jobs: find a pulse map, find a truth-track-like particle, and calculate distance from a point to a line.

The distance formula is vector geometry: subtract the track point from the DOM point, cross with the unit direction vector, and take the length.


In [ ]:
def find_pulse_key(frame):
    for key in ['SplitInIcePulses', 'SplitInIceDSTPulses', 'SRTInIcePulses', 'InIcePulses', 'OfflinePulses']:
        if key in frame:
            return key
    return None

def pulsed_omkeys(frame):
    pulse_key = find_pulse_key(frame)
    if pulse_key is None:
        return pulse_key, []
    pulse_map = dataclasses.I3RecoPulseSeriesMap.from_frame(frame, pulse_key)
    return pulse_key, [omkey for omkey, pulses in pulse_map if len(pulses) > 0 and is_inice_dom(omkey)]

def looks_like_particle(obj):
    return hasattr(obj, 'pos') and hasattr(obj, 'dir')

def find_truth_particle(frame):
    preferred_keys = ['MCPrimary', 'MostEnergeticMuon', 'MostEnergeticInIceMuon', 'MCMuon', 'PolyplopiaPrimary']
    for key in preferred_keys:
        if key in frame and looks_like_particle(frame[key]):
            return key, frame[key]

    # Fall back to any particle-like key with a useful name.
    for key in frame.keys():
        lower = key.lower()
        if any(word in lower for word in ['mc', 'truth', 'primary', 'muon']):
            try:
                if looks_like_particle(frame[key]):
                    return key, frame[key]
            except Exception:
                pass
    return None, None

def particle_position(particle):
    return np.array([particle.pos.x, particle.pos.y, particle.pos.z], dtype=float)

def particle_direction_unit_vector(particle):
    zenith = particle.dir.zenith
    azimuth = particle.dir.azimuth
    direction = np.array([sin(zenith) * cos(azimuth), sin(zenith) * sin(azimuth), cos(zenith)], dtype=float)
    return direction / np.linalg.norm(direction)

def distance_from_dom_to_track(dom_xyz, particle):
    track_point = particle_position(particle)
    track_direction = particle_direction_unit_vector(particle)
    return np.linalg.norm(np.cross(dom_xyz - track_point, track_direction))

print('Defined pulse, truth-particle, and distance helpers.')


## Build the lateral-distance table

We now loop through simulation Physics frames. For each event, we find pulsed DOMs and a truth track. If both exist, we calculate one distance per pulsed DOM.

The diagnostic printout is part of the lesson: if the table is empty, the diagnostics tell us whether the problem was frames, pulses, or truth keys.


In [ ]:
MAX_PHYSICS_FRAMES = 500
MIN_HIT_DOMS = 1

rows = []
physics_seen = 0
events_with_pulses = 0
events_with_truth = 0
first_pulse_key = None
first_truth_key = None
interesting_keys = []

i3_file = dataio.I3File(str(SIM_FILE))
while i3_file.more() and physics_seen < MAX_PHYSICS_FRAMES:
    frame = i3_file.pop_frame()
    if frame_stop(frame) != 'Physics':
        continue

    physics_seen += 1
    if not interesting_keys:
        interesting_keys = [key for key in frame.keys() if any(word in key.lower() for word in ['pulse', 'mc', 'muon', 'primary', 'truth'])]

    pulse_key, omkeys = pulsed_omkeys(frame)
    if pulse_key is not None:
        first_pulse_key = first_pulse_key or pulse_key
    if len(omkeys) < MIN_HIT_DOMS:
        continue
    events_with_pulses += 1

    truth_key, truth_particle = find_truth_particle(frame)
    if truth_particle is None:
        continue
    events_with_truth += 1
    first_truth_key = first_truth_key or truth_key

    event_id = frame['I3EventHeader'].event_id if 'I3EventHeader' in frame else None
    for omkey in omkeys:
        distance_m = distance_from_dom_to_track(inice_positions[omkey], truth_particle)
        rows.append({'event_id': event_id, 'string': omkey.string, 'om': omkey.om, 'truth_key': truth_key, 'distance_m': distance_m, 'event_hit_doms': len(omkeys)})

i3_file.close()
distance_table = pd.DataFrame(rows, columns=['event_id', 'string', 'om', 'truth_key', 'distance_m', 'event_hit_doms'])

print('Physics frames inspected:', physics_seen)
print('Events with enough pulsed DOMs:', events_with_pulses)
print('Events with a truth particle:', events_with_truth)
print('First pulse key found:', first_pulse_key)
print('First truth key found:', first_truth_key)
print('Interesting keys from first Physics frame:', interesting_keys[:30])
distance_table.head()


## Plot the distances

If the table is empty, do not panic. That usually means the chosen file does not have the truth key we expected, or the pulse key list needs to be adjusted. The previous cell prints clues.


In [ ]:
if distance_table.empty:
    print('No lateral-distance rows were built. Look at the diagnostic output above.')
else:
    distance_table['distance_m'].plot.hist(bins=60)
    plt.xlabel('DOM lateral distance to truth track [m]')
    plt.ylabel('pulsed DOMs')
    print('Each histogram entry is one pulsed DOM, not one event.')


## Practice

1. Increase `MIN_HIT_DOMS` to 10 after you get the first successful plot.
2. Print `distance_table.groupby('event_id')['distance_m'].median()` to get one value per event.
3. Try changing the preferred truth keys in `find_truth_particle`.
4. Explain in words what `np.cross(dom_xyz - track_point, track_direction)` is doing geometrically.
